# Task 23 — Day 2 (29/10): Capstone — An End-to-End Production Project



---

## Overview

Day 1 (Phases 1–8) taught each production tool separately on MNIST. Today's capstone puts them together on **one model, end to end**, the way a real team would ship it:

**Model:** a **sentiment classifier** (IMDB movie reviews, Task 21 style). Clients send **raw review text**, the model returns the probability that the review is positive. `TextVectorization` lives **inside** the exported model, so there is no preprocessing on the client.

| Step | Capstone item (README Phase 8, Task 11) | Section |
|---|---|---|
| 1 | Export a SavedModel **with preprocessing inside** + post-export self-test | C1 |
| 2 | Serve with **TF Serving in Docker**, **REST + gRPC** clients, benchmarked with Task 18 rules | C2 |
| 3 | Train **v2 with `MirroredStrategy`** and report the speed-up vs one device | C3 |
| 4 | Deploy **v2**, **canary** it to 10 % of users, promote, **roll back**, keep the logs | C4 |
| 5 | Enable **batching** and report throughput / latency before and after | C5 |
| 6 | Convert to **TFLite** (float16 and full-integer) and compare size / accuracy / latency | C6 |

The capstone plans and checklists (cost/latency/scale plan, security and teardown checklists), the exercises and the demo are the **30/10** slot.

**Runs on any machine:** with internet the real IMDB reviews are used (otherwise a learnable synthetic stand-in); with Docker Desktop running the serving cells run live (otherwise they print *SKIPPED*); without a GPU the CPU is split into logical devices so `MirroredStrategy` still runs for real. Generated files go into `task23_capstone_work/`.

## 0. Setup

Device configuration must run before TensorFlow touches a device, so run this cell first after every kernel restart.

In [1]:
import os, sys, json, time, shutil, socket, subprocess, threading, textwrap, random, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")      # hide INFO/WARNING spam from the C++ runtime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import absl.logging
tf.get_logger().setLevel("ERROR"); absl.logging.set_verbosity(absl.logging.ERROR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- Device setup (must happen before any op runs) -------------------------
physical_gpus = tf.config.list_physical_devices("GPU")
if physical_gpus:
    # Phase 6: split GPU 0 into two 2 GiB virtual GPUs so multi-device code can be tested on one card
    tf.config.set_logical_device_configuration(
        physical_gpus[0],
        [tf.config.LogicalDeviceConfiguration(memory_limit=2048),
         tf.config.LogicalDeviceConfiguration(memory_limit=2048)])
    DEVICES = [d.name for d in tf.config.list_logical_devices("GPU")]
else:
    # No GPU: split the CPU into 4 logical devices -> MirroredStrategy etc. still run for real
    cpu = tf.config.list_physical_devices("CPU")[0]
    tf.config.set_logical_device_configuration(cpu, [tf.config.LogicalDeviceConfiguration()] * 4)
    DEVICES = [d.name for d in tf.config.list_logical_devices("CPU")]

HAS_GPU = bool(physical_gpus)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| Python:", sys.version.split()[0])
print("Physical GPUs:", physical_gpus)
print("Logical compute devices used in this notebook:", DEVICES)

WORK = Path("task23_capstone_work").resolve(); WORK.mkdir(exist_ok=True)
SERVE_DIR = WORK / "serving"; SERVE_DIR.mkdir(exist_ok=True)   # mounted into TF Serving as /models
SCRIPTS = WORK / "scripts"; SCRIPTS.mkdir(exist_ok=True)
print("Work dir:", WORK)

TensorFlow: 2.22.0-rc0 | Keras: 3.16.0.dev2026102803 | Python: 3.14.6
Physical GPUs: []
Logical compute devices used in this notebook: ['/device:CPU:0', '/device:CPU:1', '/device:CPU:2', '/device:CPU:3']
Work dir: C:\Users\HP\Documents\Task23\task23_capstone_work


In [2]:
# ---- Small utilities reused everywhere ------------------------------------
def benchmark(fn, warmup=2, repeats=5):
    '''Task 18 discipline: warm-up calls excluded, >=5 timed repeats, return (mean, std) in seconds.'''
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return float(np.mean(times)), float(np.std(times))

def fmt_ms(mean_std):
    m, s = mean_std
    return f"{m*1000:.2f} ± {s*1000:.2f} ms"

def run_py(script_path, args=(), env_extra=None, timeout=900):
    '''Run a helper script in a fresh Python process (needed whenever TF must be configured differently).'''
    env = os.environ.copy(); env.update(env_extra or {})
    r = subprocess.run([sys.executable, str(script_path), *map(str, args)], capture_output=True,
                       text=True, env=env, timeout=timeout)
    return r.returncode, r.stdout, r.stderr

def tail(text, n=15):
    lines = [l for l in text.splitlines() if l.strip() and "absl::InitializeLog" not in l]
    return "\n".join(lines[-n:])

def dir_size_mb(path):
    p = Path(path)
    if p.is_file():
        return p.stat().st_size / 1e6
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6

def tree(path, prefix=""):
    '''Print a directory tree with file sizes (for the SavedModel anatomy).'''
    path = Path(path)
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, e in enumerate(entries):
        last = i == len(entries) - 1
        size = f"  ({e.stat().st_size/1024:.1f} KB)" if e.is_file() else "/"
        print(prefix + ("└── " if last else "├── ") + e.name + size)
        if e.is_dir():
            tree(e, prefix + ("    " if last else "│   "))

### 0.1 Toolkit reused from Day 1

The helpers built in Phases 1–7, collected in one cell: the self-testing `export_model`, the TF Serving controller with REST/gRPC clients and typed errors, the load tester, the TFLite wrapper, and the epoch timer.

In [3]:
def signature_io(path, signature="serving_default"):
    '''Return (input_name, output_name, input_spec) of a SavedModel signature.'''
    sig = tf.saved_model.load(str(path)).signatures[signature]
    in_name, in_spec = next(iter(sig.structured_input_signature[1].items()))
    out_name = next(iter(sig.structured_outputs.keys()))
    return in_name, out_name, in_spec

def export_model(model, name, version, X_check, base_dir=SERVE_DIR, atol=1e-5, verbose=True):
    '''Export `model` to base_dir/name/<version:04d>, reload it and assert predictions match.'''
    path = Path(base_dir) / name / f"{int(version):04d}"
    if path.exists():
        shutil.rmtree(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if hasattr(model, "export"):
        model.export(str(path), verbose=False)          # Keras 3 / tf.keras >= 2.13
    else:
        tf.saved_model.save(model, str(path))
    # ---- post-export self-test ----
    reloaded = tf.saved_model.load(str(path))
    sig = reloaded.signatures["serving_default"]
    in_name, in_spec = next(iter(sig.structured_input_signature[1].items()))
    x = tf.constant(X_check, dtype=in_spec.dtype)
    served = next(iter(sig(**{in_name: x}).values())).numpy()
    local = np.asarray(model(x, training=False))
    max_diff = float(np.max(np.abs(served - local)))
    assert np.allclose(served, local, atol=atol), f"Self-test FAILED for {path}: max diff {max_diff}"
    if verbose:
        print(f"Exported {path.relative_to(WORK)}  |  {dir_size_mb(path):.2f} MB  |  self-test OK (max diff {max_diff:.1e})")
    return path

def docker_ready():
    if shutil.which("docker") is None:
        return False, "docker CLI not found"
    try:
        r = subprocess.run(["docker", "info"], capture_output=True, text=True, timeout=30)
        if r.returncode != 0:
            return False, "Docker daemon not running (start Docker Desktop)"
        r = subprocess.run(["docker", "image", "inspect", "tensorflow/serving"], capture_output=True, text=True, timeout=30)
        if r.returncode != 0:
            return False, "image missing - run:  docker pull tensorflow/serving"
        return True, "ok"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

def free_port():
    with socket.socket() as s:
        s.bind(("localhost", 0)); return s.getsockname()[1]

try:
    import grpc
    from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc
    HAS_GRPC = True
except ImportError:
    HAS_GRPC = False

import requests
from concurrent.futures import ThreadPoolExecutor

class ServingError(Exception): pass

class ModelNotFoundError(ServingError): pass

class InvalidRequestError(ServingError): pass        # wrong shape / dtype / signature

class ServerUnavailableError(ServingError): pass

class TFServing:
    '''Start/stop a tensorflow/serving container that mounts SERVE_DIR at /models.'''
    def __init__(self, container="tf_serving_task23", rest_port=8501, grpc_port=8500):
        self.container, self.rest_port, self.grpc_port = container, rest_port, grpc_port
        self._channel = None

    # ---- lifecycle ----
    def start(self, model_name=None, config_file=None, extra_args=(), timeout=120):
        self.stop(quiet=True)
        cmd = ["docker", "run", "-d", "--rm", "--name", self.container,
               "-p", f"{self.grpc_port}:8500", "-p", f"{self.rest_port}:8501",
               "-v", f"{SERVE_DIR}:/models"]
        if model_name:
            cmd += ["-e", f"MODEL_NAME={model_name}"]
        cmd += ["tensorflow/serving"]
        if config_file:
            cmd += [f"--model_config_file=/models/{config_file}", "--model_config_file_poll_wait_seconds=60"]
        cmd += ["--file_system_poll_wait_seconds=1", *extra_args]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise ServerUnavailableError(r.stderr.strip())
        self.start_cmd = cmd
        return self

    def wait_available(self, model, versions=None, timeout=120):
        t0 = time.time()
        while time.time() - t0 < timeout:
            try:
                st = self.status(model)
                avail = {v for v, s in st if s == "AVAILABLE"}
                if avail and (versions is None or set(versions) <= avail):
                    return st
            except Exception:
                pass
            time.sleep(0.5)
        raise ServerUnavailableError(f"{model} not AVAILABLE after {timeout}s\n" + self.logs(20))

    def stop(self, quiet=False):
        if self._channel is not None:
            self._channel.close(); self._channel = None
        r = subprocess.run(["docker", "rm", "-f", self.container], capture_output=True, text=True)
        if not quiet:
            print("stopped" if r.returncode == 0 else "nothing to stop")

    def logs(self, n=30):
        r = subprocess.run(["docker", "logs", "--tail", str(n), self.container], capture_output=True, text=True)
        return r.stdout + r.stderr

    # ---- REST ----
    def url(self, model, version=None, verb=""):
        u = f"http://localhost:{self.rest_port}/v1/models/{model}"
        if version is not None:
            u += f"/versions/{version}"
        return u + (f":{verb}" if verb else "")

    def status(self, model):
        r = requests.get(self.url(model), timeout=5)
        r.raise_for_status()
        return [(int(v["version"]), v["state"]) for v in r.json()["model_version_status"]]

    def rest_predict(self, X, model, version=None, signature="serving_default", timeout=30):
        payload = {"signature_name": signature, "instances": np.asarray(X).tolist()}
        try:
            r = requests.post(self.url(model, version, "predict"), json=payload, timeout=timeout)
        except requests.ConnectionError as e:
            raise ServerUnavailableError(str(e)) from None
        if r.status_code == 404:
            raise ModelNotFoundError(r.json().get("error", r.text))
        if r.status_code >= 400:
            raise InvalidRequestError(r.json().get("error", r.text) if r.headers.get("content-type","").startswith("application/json") else r.text)
        return np.array(r.json()["predictions"])

    # ---- gRPC ----
    def grpc_stub(self):
        if self._channel is None:
            opts = [("grpc.max_send_message_length", 512 * 1024 * 1024),
                    ("grpc.max_receive_message_length", 512 * 1024 * 1024)]
            self._channel = grpc.insecure_channel(f"localhost:{self.grpc_port}", options=opts)
            self._stub = prediction_service_pb2_grpc.PredictionServiceStub(self._channel)
        return self._stub

    @staticmethod
    def make_request(X, model, input_name, version=None, signature="serving_default"):
        request = predict_pb2.PredictRequest()
        request.model_spec.name = model
        request.model_spec.signature_name = signature
        if version is not None:
            request.model_spec.version.value = int(version)
        request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X))
        return request

    def grpc_predict(self, X, model, input_name, output_name, version=None, signature="serving_default",
                     timeout=10.0, return_version=False):
        request = self.make_request(X, model, input_name, version, signature)
        try:
            response = self.grpc_stub().Predict(request, timeout=timeout)   # synchronous, blocking call
        except grpc.RpcError as e:
            code = e.code()
            if code == grpc.StatusCode.NOT_FOUND:
                raise ModelNotFoundError(e.details()) from None
            if code == grpc.StatusCode.INVALID_ARGUMENT:
                raise InvalidRequestError(e.details()) from None
            if code in (grpc.StatusCode.UNAVAILABLE, grpc.StatusCode.DEADLINE_EXCEEDED):
                raise ServerUnavailableError(e.details()) from None
            raise ServingError(f"{code}: {e.details()}") from None
        y = tf.make_ndarray(response.outputs[output_name])
        return (y, response.model_spec.version.value) if return_version else y

def assign_version(user_id, canary_share=0.10):
    import hashlib
    h = int(hashlib.md5(str(user_id).encode()).hexdigest(), 16) % 1000
    return 2 if h < canary_share * 1000 else 1

def write_batching_config(fname, timeout_us, max_batch=64):
    (SERVE_DIR / fname).write_text(textwrap.dedent(f'''
        max_batch_size {{ value: {max_batch} }}
        batch_timeout_micros {{ value: {timeout_us} }}
        max_enqueued_batches {{ value: 1000 }}
        num_batch_threads {{ value: {os.cpu_count() or 4} }}'''))

def load_test(n_clients=50, req_per_client=20, model=None, payload_fn=None):
    '''n_clients threads, each sending req_per_client single-item requests; failed requests are counted, not fatal.'''
    model = model or model_name
    payload_fn = payload_fn or (lambda i: X_test[i % len(X_test):i % len(X_test) + 1].tolist())
    def client(cid):
        s = requests.Session(); ok, errs = [], 0
        for i in range(req_per_client):
            t0 = time.perf_counter()
            try:
                r = s.post(serving.url(model, verb="predict"),
                           json={"instances": payload_fn(cid * req_per_client + i)}, timeout=30)
                r.raise_for_status(); ok.append(time.perf_counter() - t0)
            except requests.RequestException:
                errs += 1
        return ok, errs
    lat, errors = [], 0
    t0 = time.perf_counter()
    with ThreadPoolExecutor(n_clients) as ex:
        for ok, e in ex.map(client, range(n_clients)):
            lat.extend(ok); errors += e
    wall = time.perf_counter() - t0
    lat = np.array(lat) * 1000 if lat else np.array([np.nan])
    return {"throughput_rps": len(lat) / wall, "p50_ms": np.percentile(lat, 50),
            "p95_ms": np.percentile(lat, 95), "p99_ms": np.percentile(lat, 99), "errors": errors}

class TFLiteModel:
    '''Thin wrapper around the TFLite interpreter that handles int8 input/output quantization.'''
    def __init__(self, tflite_bytes, num_threads=1):
        self.interp = tf.lite.Interpreter(model_content=tflite_bytes, num_threads=num_threads)
        self.inp = self.interp.get_input_details()[0]; self.out = self.interp.get_output_details()[0]
        self.size_kb = len(tflite_bytes) / 1024
        self._bs = None

    def predict(self, X):
        if self._bs != len(X):
            self.interp.resize_tensor_input(self.inp["index"], [len(X), *self.inp["shape"][1:]])
            self.interp.allocate_tensors(); self._bs = len(X)
            self.inp = self.interp.get_input_details()[0]; self.out = self.interp.get_output_details()[0]
        x = np.asarray(X)
        if self.inp["dtype"] in (np.int8, np.uint8):              # quantize input
            scale, zero = self.inp["quantization"]
            x = np.round(x / scale + zero).clip(-128 if self.inp["dtype"] == np.int8 else 0,
                                                127 if self.inp["dtype"] == np.int8 else 255)
        self.interp.set_tensor(self.inp["index"], x.astype(self.inp["dtype"]))
        self.interp.invoke()
        y = self.interp.get_tensor(self.out["index"])
        if self.out["dtype"] in (np.int8, np.uint8):              # dequantize output
            scale, zero = self.out["quantization"]
            y = (y.astype(np.float32) - zero) * scale
        return y

    def accuracy(self, X, y, batch=500):
        preds = np.concatenate([self.predict(X[i:i+batch]) for i in range(0, len(X), batch)])
        return float((preds.argmax(1) == y).mean())

def convert(saved_model_path, tweak=None):
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_path))
    if tweak:
        tweak(converter)
    return converter.convert()

def t_fp16(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]

class EpochTimer(keras.callbacks.Callback):
    def on_epoch_begin(self, e, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, e, logs=None): self.times.append(time.perf_counter() - self.t0)
    def on_train_begin(self, logs=None): self.times = []


DOCKER_OK, DOCKER_MSG = docker_ready()
serving = TFServing()
SERVING_UP = DOCKER_OK
print("Docker + tensorflow/serving available:", DOCKER_OK, "-", DOCKER_MSG)
print("gRPC client libs available:", HAS_GRPC)
TWO = DEVICES[:2]

Docker + tensorflow/serving available: True - ok
gRPC client libs available: True


---
# Capstone

**Model:** the **sentiment / text classifier** track (Task 21 style) — raw review text in, probability of "positive" out, with `TextVectorization` **inside** the exported model so clients send plain strings.


### C1. Data, model v1, export with preprocessing inside

Reviews are decoded back to text so the model sees exactly what a client would send. `TextVectorization` is adapted on the training text, then wrapped together with the core network into one model whose input is a **string** tensor. The export helper reloads the SavedModel and checks that it gives the same probabilities as the in-memory model.

In [4]:
def load_reviews(n_train=10000, n_test=2500, seed=SEED):
    try:
        (xtr, ytr), (xte, yte) = keras.datasets.imdb.load_data()
        word_index = keras.datasets.imdb.get_word_index()
        inv = {i + 3: w for w, i in word_index.items()}
        decode = lambda seq: " ".join(inv.get(i, "") for i in seq if i >= 4)
        rng = np.random.default_rng(seed)
        itr = rng.choice(len(xtr), n_train, replace=False); ite = rng.choice(len(xte), n_test, replace=False)
        return ([decode(xtr[i]) for i in itr], ytr[itr].astype("float32"),
                [decode(xte[i]) for i in ite], yte[ite].astype("float32"), "real IMDB reviews")
    except Exception as e:
        print("IMDB download failed ->", type(e).__name__, "- using synthetic reviews")
        rng = np.random.default_rng(seed)
        pos = "great excellent wonderful loved brilliant superb moving enjoyable fantastic masterpiece".split()
        neg = "awful boring terrible hated waste dull poor worst disappointing mess".split()
        neutral = ("the movie film plot actor story scene director acting music character was it this and "
                   "of a to really quite very some ending script").split()
        def make(n):
            texts, labels = [], []
            for _ in range(n):
                y = rng.integers(0, 2); words = list(rng.choice(neutral, 25))
                k = rng.integers(2, 6)
                words += list(rng.choice(pos if y else neg, k)) + list(rng.choice(neg if y else pos, rng.integers(0, k)))
                rng.shuffle(words); texts.append(" ".join(words)); labels.append(float(y))
            return texts, np.array(labels, "float32")
        a, b = make(n_train); c, d = make(n_test)
        return a, b, c, d, "synthetic reviews"

txt_train, ys_train, txt_test, ys_test, TEXT_SOURCE = load_reviews()
print(TEXT_SOURCE, "| train", len(txt_train), "| test", len(txt_test), "| positive share", ys_train.mean().round(3))
print("example:", txt_train[0][:160], "... ->", ys_train[0])

VOCAB, SEQ_LEN = 10000, 200
vectorizer = keras.layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=SEQ_LEN, name="vectorizer")
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(txt_train).batch(512))

def build_core(embed_dim=32, hidden=16, name="sentiment_core"):
    '''The part that runs on token ids - this is what goes to TFLite.'''
    return keras.Sequential([keras.Input((SEQ_LEN,), dtype="int32", name="token_ids"),
                             keras.layers.Embedding(VOCAB, embed_dim),
                             keras.layers.GlobalAveragePooling1D(),
                             keras.layers.Dense(hidden, activation="relu"),
                             keras.layers.Dense(1, activation="sigmoid")], name=name)

def build_full(vec, core, name):
    text_in = keras.Input(shape=(), dtype="string", name="text")
    ids = keras.layers.Lambda(lambda t: tf.cast(t, tf.int32), name="to_int32")(vec(text_in))
    model = keras.Model(text_in, core(ids), name=name)
    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

def text_ds(texts, labels, batch=64, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    return (ds.shuffle(len(texts), seed=SEED) if shuffle else ds).batch(batch).prefetch(tf.data.AUTOTUNE)

tf.random.set_seed(SEED)
core_v1 = build_core()
sent_v1 = build_full(vectorizer, core_v1, "sentiment_v1")
sent_v1.fit(text_ds(txt_train, ys_train), epochs=4, validation_data=text_ds(txt_test, ys_test, shuffle=False), verbose=2)
s1_acc = sent_v1.evaluate(text_ds(txt_test, ys_test, shuffle=False), verbose=0)[1]

CAP = "sentiment"
probe_texts = np.array(["What a wonderful, moving film - I loved it.",
                        "Boring plot and terrible acting, a total waste of time.",
                        txt_test[0][:300]])
cap_v1_path = export_model(sent_v1, CAP, 1, probe_texts)
cap_in, cap_out, cap_spec = signature_io(cap_v1_path)
print(f"v1 test accuracy {s1_acc:.4f} | signature input {cap_in!r} {cap_spec.dtype.name} {tuple(cap_spec.shape)} -> {cap_out!r}")
print("Raw strings in, probabilities out:", np.round(sent_v1.predict(tf.constant(probe_texts[:2]), verbose=0).ravel(), 3))

       0/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   16384/17464789 ━━━━━━━━━━━━━━━━━━━━ 5:38 19us/step

   49152/17464789 ━━━━━━━━━━━━━━━━━━━━ 3:36 12us/step

   81920/17464789 ━━━━━━━━━━━━━━━━━━━━ 3:15 11us/step

  114688/17464789 ━━━━━━━━━━━━━━━━━━━━ 2:52 10us/step

...

Epoch 4/4


157/157 - 2s - 14ms/step - accuracy: 0.8846 - loss: 0.2998 - val_accuracy: 0.8400 - val_loss: 0.3595


Exported serving\sentiment\0001  |  1.51 MB  |  self-test OK (max diff 6.0e-08)


v1 test accuracy 0.8400 | signature input 'text' string (None,) -> 'output_0'


Raw strings in, probabilities out: [0.791 0.13 ]


### C2. Serve with TF Serving — REST + gRPC, benchmarked

In [5]:
def to_grpc_strings(texts):
    return np.array([t.encode("utf-8") for t in texts], dtype=object)

cap_rows = []
if SERVING_UP:
    shutil.rmtree(SERVE_DIR / CAP / "0002", ignore_errors=True)
    serving.start(model_name=CAP); serving.wait_available(CAP)
    r_rest = serving.rest_predict(list(probe_texts), CAP).ravel()
    print("REST:", np.round(r_rest, 4))
    if HAS_GRPC:
        r_grpc = serving.grpc_predict(to_grpc_strings(probe_texts), CAP, cap_in, cap_out).ravel()
        print("gRPC:", np.round(r_grpc, 4), "| equal:", np.allclose(r_rest, r_grpc, atol=1e-5))
    for bs in (1, 32, 256):
        batch = txt_test[:bs]
        row = {"batch": bs,
               "REST payload KB": len(json.dumps({"instances": batch}).encode()) / 1e3,
               "REST latency": fmt_ms(benchmark(lambda: serving.rest_predict(batch, CAP)))}
        if HAS_GRPC:
            row["gRPC payload KB"] = TFServing.make_request(to_grpc_strings(batch), CAP, cap_in).ByteSize() / 1e3
            row["gRPC latency"] = fmt_ms(benchmark(lambda: serving.grpc_predict(to_grpc_strings(batch), CAP, cap_in, cap_out)))
        cap_rows.append(row)
    display(pd.DataFrame(cap_rows).round(1))
    print("For TEXT the payloads are almost the same size (strings are strings in both protocols) -> "
          "gRPC's advantage here is lower serialization overhead, not payload size.")
else:
    print("SKIPPED live serving of the capstone model (no Docker). Local check of the exported signature:")
    sig = tf.saved_model.load(str(cap_v1_path)).signatures["serving_default"]
    print(np.round(sig(tf.constant(probe_texts))[cap_out].numpy().ravel(), 4))

REST: [0.7912 0.1304 0.8918]
gRPC: [0.7912 0.1304 0.8918] | equal: True


,batch,REST payload KB,REST latency,gRPC payload KB,gRPC latency
0,1,2.3,16.91 ± 6.75 ms,2.3,4.92 ± 0.28 ms
1,32,38.6,17.35 ± 2.02 ms,38.6,46.96 ± 5.70 ms
2,256,300.3,53.89 ± 1.77 ms,300.0,32.98 ± 17.40 ms


For TEXT the payloads are almost the same size (strings are strings in both protocols) -> gRPC's advantage here is lower serialization overhead, not payload size.


### C3. Train v2 with `MirroredStrategy` and measure the speed-up

Strings can't be all-reduced across replicas (TF tried to `AddN` string tensors when the `TextVectorization` layer was inside the mirrored model), so — following the Phase 6 placement rule — text is vectorised **on the CPU inside `tf.data`**, only the numeric core is replicated, and the vectoriser is re-attached afterwards for the exported serving model.

In [6]:
# Placement rule from Phase 6: string preprocessing on the CPU (tf.data), only the numeric core is replicated
def ids_ds(texts, labels, batch, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle: ds = ds.shuffle(len(texts), seed=SEED)
    return ds.batch(batch).map(lambda t, y: (tf.cast(vectorizer(t), tf.int32), y),
                               num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def train_sentiment(strategy, epochs=4, embed_dim=48, hidden=32, label=""):
    with strategy.scope():
        tf.random.set_seed(SEED + 7)
        core = build_core(embed_dim, hidden, name="sentiment_core_v2")
        # linear scaling rule from Phase 7.8: global batch x R -> learning rate x R
        core.compile(loss="binary_crossentropy", metrics=["accuracy"],
                     optimizer=keras.optimizers.Adam(1e-3 * strategy.num_replicas_in_sync))
    timer = EpochTimer()
    gb = 64 * strategy.num_replicas_in_sync
    core.fit(ids_ds(txt_train, ys_train, gb), epochs=epochs, callbacks=[timer], verbose=0)
    model = build_full(vectorizer, core, "sentiment_v2")          # re-attach preprocessing for serving
    acc = model.evaluate(text_ds(txt_test, ys_test, shuffle=False), verbose=0)[1]
    ep = timer.times[1:]
    return model, {"run": label, "replicas": strategy.num_replicas_in_sync, "global_batch": gb,
                   "epoch_s": np.mean(ep), "epoch_std": np.std(ep), "test_acc": acc}

_, single_row = train_sentiment(tf.distribute.OneDeviceStrategy(DEVICES[0]), label="one device")
sent_v2, mirror_row = train_sentiment(tf.distribute.MirroredStrategy(TWO, cross_device_ops=tf.distribute.ReductionToOneDevice()),
                                     label=f"MirroredStrategy x{len(TWO)}")
cap_train_df = pd.DataFrame([single_row, mirror_row])
cap_train_df["speedup"] = (cap_train_df["epoch_s"].iloc[0] / cap_train_df["epoch_s"]).round(2)
display(cap_train_df.round(4))
s2_acc = mirror_row["test_acc"]
cap_v2_path = export_model(sent_v2, CAP + "_staging", 2, probe_texts, base_dir=WORK)   # staged, not yet live

,run,replicas,global_batch,epoch_s,epoch_std,test_acc,speedup
0,one device,1,64,2.9681,0.0440,0.8180,1.00
1,MirroredStrategy x2,2,128,2.9503,0.1751,0.8392,1.01


Exported sentiment_staging\0002  |  4.09 MB  |  self-test OK (max diff 6.0e-08)


### C4. Deploy v2, canary 10 %, roll back

In [7]:
fp_s1 = sent_v1.predict(tf.constant(probe_texts[:1]), verbose=0).ravel()[0]
fp_s2 = sent_v2.predict(tf.constant(probe_texts[:1]), verbose=0).ravel()[0]
cap_log_lines = []
if SERVING_UP:
    # 1) deploy: copy staged 0002 into the live model directory
    shutil.copytree(cap_v2_path, SERVE_DIR / CAP / "0002", dirs_exist_ok=True)
    # 2) canary: both versions loaded, 10 % of users pinned to v2
    (SERVE_DIR / "sentiment_ab.config").write_text(textwrap.dedent(f'''
        model_config_list {{
          config {{ name: "{CAP}" base_path: "/models/{CAP}" model_platform: "tensorflow"
                   model_version_policy {{ specific {{ versions: 1 versions: 2 }} }} }}
        }}'''))
    serving.start(config_file="sentiment_ab.config"); serving.wait_available(CAP, versions=[1, 2])
    rng = np.random.default_rng(1); log = []
    for _ in range(300):
        user = int(rng.integers(0, 10**6)); i = int(rng.integers(0, len(txt_test))); v = assign_version(user)
        t0 = time.perf_counter(); p = serving.rest_predict([txt_test[i]], CAP, version=v).ravel()[0]
        log.append({"version": v, "ms": (time.perf_counter() - t0) * 1000, "correct": int((p > .5) == ys_test[i])})
    canary = pd.DataFrame(log).groupby("version").agg(requests=("ms", "size"), accuracy=("correct", "mean"),
                                                     p95_ms=("ms", lambda s: s.quantile(.95))).round(3)
    display(canary)
    cap_log_lines += [l for l in serving.logs(80).splitlines() if "sentiment" in l and ("version" in l.lower() or "load" in l.lower())]
    # 3) promote v2 to 100 % ('latest' policy), then 4) roll back by removing 0002
    serving.start(model_name=CAP); serving.wait_available(CAP)
    shutil.rmtree(SERVE_DIR / CAP / "0002")
    for _ in range(60):
        st = dict(serving.status(CAP))
        if st.get(1) == "AVAILABLE" and st.get(2) != "AVAILABLE": break
        time.sleep(0.5)
    p = serving.rest_predict(list(probe_texts[:1]), CAP).ravel()[0]
    print("After rollback status:", serving.status(CAP), "-> answered by",
          "v1" if abs(p - fp_s1) < abs(p - fp_s2) else "v2")
    cap_log_lines += [l for l in serving.logs(40).splitlines() if "version" in l.lower()]
    print("\n--- server log excerpt ---\n" + "\n".join(cap_log_lines[-15:]))
else:
    print(f"SKIPPED live canary/rollback (no Docker). Offline comparison: v1 test acc {s1_acc:.4f} vs v2 {s2_acc:.4f}; "
          f"user routing share to v2 = {np.mean([assign_version(u) == 2 for u in range(20000)]):.1%}")

,requests,accuracy,p95_ms
version,,,
1,262,0.840,31.750
2,38,0.816,31.308


After rollback status: [(2, 'END'), (1, 'AVAILABLE')] -> answered by v1

--- server log excerpt ---
I0000 00:00:1793254885.546397      56 loader.cc:220] Running initialization op on SavedModel bundle at path: /models/sentiment/0001
I0000 00:00:1793254885.983863      57 loader_harness.cc:108] Successfully loaded servable version {name: sentiment version: 2}
I0000 00:00:1793254886.228027      56 loader_harness.cc:108] Successfully loaded servable version {name: sentiment version: 1}
I0000 00:00:1793254894.950050      57 basic_manager.cc:786] Successfully reserved resources to load servable {name: sentiment version: 2}
I0000 00:00:1793254894.950234      57 loader_harness.cc:70] Approving load for servable version {name: sentiment version: 2}
I0000 00:00:1793254894.950364      57 loader_harness.cc:78] Loading servable version {name: sentiment version: 2}
I0000 00:00:1793254895.782024      57 loader_harness.cc:108] Successfully loaded servable version {name: sentiment version: 2}
I0000 00:0

### C5. Batching before / after

In [8]:
def text_load_test(n_clients=50, req_per_client=10):
    return load_test(n_clients, req_per_client, model=CAP, payload_fn=lambda i: [txt_test[i % len(txt_test)]])

if SERVING_UP:
    cap_batch = []
    for label, extra in [("batching OFF", []),
                         ("batching ON (2 ms)", ["--enable_batching=true", "--batching_parameters_file=/models/batching.config"])]:
        write_batching_config("batching.config", 2000)
        serving.start(model_name=CAP, extra_args=extra); serving.wait_available(CAP)
        text_load_test(5, 3)                                    # warm-up
        cap_batch.append({"config": label, **text_load_test()})
    display(pd.DataFrame(cap_batch).set_index("config").round(1))
else:
    print("SKIPPED live batching test (no Docker).")

,throughput_rps,p50_ms,p95_ms,p99_ms,errors
config,,,,,
batching OFF,273.6,130.6,213.0,244.9,0
batching ON (2 ms),279.8,129.5,198.4,235.4,0


### C6. TFLite: float16 and full-integer for the capstone

TFLite builtin ops don't include string processing, so the **on-device split** is: the app tokenises the text with the same vocabulary (shipped as an asset — `vocab.txt` below), and the `.tflite` model runs on token ids. (Alternative: keep `TextVectorization` in the model and enable `SELECT_TF_OPS`, which bundles a much larger runtime.)

In [9]:
CAP_TFL = WORK / "tflite_capstone"; CAP_TFL.mkdir(exist_ok=True)
(CAP_TFL / "vocab.txt").write_text("\n".join(vectorizer.get_vocabulary()), encoding="utf-8")
core_path = export_model(core_v1, "sentiment_core", 1,
                         vectorizer(tf.constant(probe_texts)).numpy().astype("int32"), base_dir=WORK / "export")
ids_test = vectorizer(tf.constant(txt_test)).numpy().astype("int32")
ids_train = vectorizer(tf.constant(txt_train[:300])).numpy().astype("int32")

def rep_text():
    for i in range(300):
        yield [ids_train[i:i+1]]

def t_cap_int8(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]; c.representative_dataset = rep_text
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]    # int8 kernels everywhere; ids stay int32

cap_variants = {}
for label, tweak in [("float32", None), ("float16", t_fp16), ("full-integer int8", t_cap_int8)]:
    try:
        cap_variants[label] = convert(core_path, tweak)
        (CAP_TFL / f"sentiment_{label.split()[0].replace('-', '_')}.tflite").write_bytes(cap_variants[label])
    except Exception as e:
        print(label, "conversion failed:", type(e).__name__, str(e)[:200])

def bin_acc(pred_fn, X, y, batch=250):
    p = np.concatenate([np.asarray(pred_fn(X[i:i+batch])).ravel() for i in range(0, len(X), batch)])
    return float(((p > 0.5) == y).mean())

cap_sm = tf.saved_model.load(str(cap_v1_path)).signatures["serving_default"]
cap_tfl_rows = [{"variant": "SavedModel (full, strings in)", "size_KB": dir_size_mb(cap_v1_path) * 1000,
                 "accuracy": bin_acc(lambda t: cap_sm(tf.constant(t))[cap_out].numpy(), np.array(txt_test), ys_test),
                 "latency_1": fmt_ms(benchmark(lambda: cap_sm(tf.constant(probe_texts[:1])), warmup=3, repeats=20))}]
for label, b in cap_variants.items():
    m = TFLiteModel(b)
    cap_tfl_rows.append({"variant": f"TFLite {label} (token ids in)", "size_KB": len(b) / 1024,
                         "accuracy": bin_acc(m.predict, ids_test, ys_test),
                         "latency_1": fmt_ms(benchmark(lambda: m.predict(ids_test[:1]), warmup=3, repeats=20))})
pd.DataFrame(cap_tfl_rows).round({"size_KB": 1, "accuracy": 4})

Exported export\sentiment_core\0001  |  1.31 MB  |  self-test OK (max diff 6.0e-08)


,variant,size_KB,accuracy,latency_1
0,"SavedModel (full, strings in)",1509.5,0.8400,2.22 ± 0.22 ms
1,TFLite float32 (token ids in),1255.2,0.8400,0.02 ± 0.00 ms
2,TFLite float16 (token ids in),629.9,0.8400,0.02 ± 0.00 ms
3,TFLite full-integer int8 (token ids in),317.0,0.8376,0.02 ± 0.01 ms


### Teardown
Stop the TF Serving container so nothing keeps running after the notebook (the cloud-side teardown checklist is part of tomorrow's plans).

In [10]:
if DOCKER_OK:
    serving.stop()
print("TF Serving container stopped (teardown).")

stopped
TF Serving container stopped (teardown).


---
## Results summary (filled from this run)

In [11]:
rows = []
rows.append(("C1 export", f"v1 test accuracy {s1_acc:.4f}; SavedModel takes raw strings; self-test passed"))
rows.append(("C2 serving", "REST + gRPC answered and matched" if SERVING_UP else "SKIPPED (Docker not running)"))
if cap_rows:
    b = pd.DataFrame(cap_rows).set_index("batch")
    rows.append(("C2 latency (batch 32)", f"REST {b.loc[32,'REST latency']} | gRPC {b.loc[32].get('gRPC latency','-')}"))
rows.append(("C3 MirroredStrategy", f"epoch {cap_train_df.loc[0,'epoch_s']:.2f}s -> {cap_train_df.loc[1,'epoch_s']:.2f}s "
             f"(x{cap_train_df.loc[1,'speedup']:.2f}); v2 test accuracy {s2_acc:.4f}"))
if SERVING_UP:
    rows.append(("C4 canary", "; ".join(f"v{v}: {int(r.requests)} req, acc {r.accuracy:.3f}" for v, r in canary.iterrows())
                 + " -> rolled back to v1"))
    cb = pd.DataFrame(cap_batch).set_index("config")
    rows.append(("C5 batching", " | ".join(f"{k}: {r.throughput_rps:.0f} req/s, p95 {r.p95_ms:.0f} ms" for k, r in cb.iterrows())))
t = pd.DataFrame(cap_tfl_rows).set_index("variant")
rows.append(("C6 TFLite", " | ".join(f"{k.split(' (')[0]}: {r.size_KB:.0f} KB, acc {r.accuracy:.4f}" for k, r in t.iterrows())))
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows, columns=["item", "result"]).set_index("item")

,result
item,
C1 export,v1 test accuracy 0.8400; SavedModel takes raw strings; self-test passed
C2 serving,REST + gRPC answered and matched
C2 latency (batch 32),REST 17.35 ± 2.02 ms | gRPC 46.96 ± 5.70 ms
C3 MirroredStrategy,epoch 2.97s -> 2.95s (x1.01); v2 test accuracy 0.8392
C4 canary,"v1: 262 req, acc 0.840; v2: 38 req, acc 0.816 -> rolled back to v1"
C5 batching,"batching OFF: 274 req/s, p95 213 ms | batching ON (2 ms): 280 req/s, p95 198 ms"
C6 TFLite,"SavedModel: 1510 KB, acc 0.8400 | TFLite float32: 1255 KB, acc 0.8400 | TFLite float16: 630 KB, acc 0.8400 | TFLite full-integer int8: 317 KB, acc 0.8376"
